PARCIAL 2 LIMPIEZA DE DATOS

[GITHUB](https://github.com/1706712022/etl-data-pipeline1706712022.git)

In [ ]:
import pandas as pd

In [ ]:
url = "https://raw.githubusercontent.com/1706712022/etl-data-pipeline1706712022/refs/heads/main/data/raw/E_inventario.csv"

In [ ]:
df_inv = pd.read_csv(url)
print("Dataset Cargado Correctamente")

Dataset Cargado Correctamente


In [ ]:
df_inv.head()

,id_inventario,id_bodega,item,cantidad,costo_unitario
0,INV5000,BOD103,Producto 45,359,192.81
1,INV5001,BOD111,Producto 2,419,153.48
2,INV5002,BOD999,Producto 120,58,104.91
3,INV5003,BOD100,Producto 42,422,$ 138.66
4,INV5004,BOD112,Producto 79,257,NaN


In [ ]:
print("Columnas del dataset");
print(df_inv.columns);

Columnas del dataset
Index(['id_inventario', 'id_bodega', 'item', 'cantidad', 'costo_unitario'], dtype='object')


In [ ]:
print("Tipos de datos del dataset");
print(df_inv.dtypes);

Tipos de datos del dataset
id_inventario     object
id_bodega         object
item              object
cantidad          object
costo_unitario    object
dtype: object


In [ ]:
print("Valores nulos del dataset");
print(df_inv.isnull().sum());

Valores nulos del dataset
id_inventario     0
id_bodega         5
item              0
cantidad          5
costo_unitario    5
dtype: int64


In [ ]:
print ("Valores duplicados del dataset");
print (df_inv.duplicated().sum());

Valores duplicados del dataset
6


NORMALIZACION DE DATOS



In [ ]:
#Quitar espacios en blanco al inicio y final de todas las celdas de texto
df_inv = df_inv.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

In [ ]:
# Quitar espacios en blanco
df_inv['id_inventario'] = df_inv['id_inventario'].str.strip()
df_inv['id_bodega'] = df_inv['id_bodega'].str.strip()
df_inv['item'] = df_inv['item'].str.strip()

VALIDACION Y SEPARACION DE DATOS : REJECTED / CURATED

In [ ]:
df_inv['motivo_rechazo'] = ""

In [ ]:
# Regla 1: ID Inventario o Bodega nulos
df_inv.loc[df_inv['id_inventario'].isna(), 'motivo_rechazo'] += "ID Inventario Nulo; "
df_inv.loc[df_inv['id_bodega'].isna(), 'motivo_rechazo'] += "ID Bodega Nulo; "

In [ ]:

# Regla 3: Bodega fuera de rango (BOD999)
df_inv.loc[df_inv['id_bodega'] == 'BOD999', 'motivo_rechazo'] += "Bodega no existente (999); "

In [ ]:
inventario_rejects = df_inv[df_inv['motivo_rechazo'] != ""].copy()

In [ ]:
df_inv['cantidad_clean'] = df_inv['cantidad'].astype(str).str.replace(r'[^\d.]', '', regex=True)
df_inv['cantidad_clean'] = pd.to_numeric(df_inv['cantidad_clean'], errors='coerce')
df_inv['cantidad_clean'] = df_inv['cantidad_clean'].fillna(0)
df_inv['cantidad_clean'] = df_inv['cantidad_clean'].astype(int)

print("Columna 'cantidad_clean' creada y limpiada.")
print(df_inv[['cantidad', 'cantidad_clean']].head())

Columna 'cantidad_clean' creada y limpiada.
  cantidad  cantidad_clean
0      359             359
1      419             419
2       58              58
3      422             422
4      257             257


In [ ]:
#SEPARACIÓN
inventario_curated = df_inv[df_inv['motivo_rechazo'] == ""].copy()
inventario_rejects = df_inv[df_inv['motivo_rechazo'] != ""].copy()

In [ ]:
print("Primeras 5 filas de inventario_curated:")
print(inventario_curated.head())

print("\nTipos de datos de inventario_curated:")
print(inventario_curated.dtypes)

Primeras 5 filas de inventario_curated:
  id_inventario id_bodega         item cantidad costo_unitario motivo_rechazo  \
0       INV5000    BOD103  Producto 45      359         192.81                  
1       INV5001    BOD111   Producto 2      419         153.48                  
3       INV5003    BOD100  Producto 42      422       $ 138.66                  
4       INV5004    BOD112  Producto 79      257            NaN                  
6       INV5006    BOD111  Producto 88      436         333.93                  

   cantidad_clean  costo_clean  
0             359       192.81  
1             419       153.48  
3             422       138.66  
4             257         0.00  
6             436       333.93  

Tipos de datos de inventario_curated:
id_inventario      object
id_bodega          object
item               object
cantidad           object
costo_unitario     object
motivo_rechazo     object
cantidad_clean      int64
costo_clean       float64
dtype: object


In [ ]:
print("Primeras filas de inventario_rejects y su motivo de rechazo:")
print(inventario_rejects[['id_inventario', 'id_bodega', 'cantidad', 'costo_unitario', 'motivo_rechazo']].head())

Primeras filas de inventario_rejects y su motivo de rechazo:
   id_inventario id_bodega  cantidad costo_unitario  \
2        INV5002    BOD999        58         104.91   
5        INV5005       NaN       488          228.6   
9        INV5009    BOD115  sin dato         277.58   
26       INV5026    BOD117       NaN          29.53   
34       INV5034    BOD999        25         338.68   

                 motivo_rechazo  
2   Bodega no existente (999);   
5              ID Bodega Nulo;   
9           Cantidad inválida;   
26          Cantidad inválida;   
34  Bodega no existente (999);   
